In [1]:
from pathlib import Path
import numpy as np
from urllib.request import urlretrieve
import pandas as pd

In [2]:
# SNIPS_DATA_BASE_URL = (
#     "https://github.com/ogrisel/slot_filling_and_intent_detection_of_SLU/blob/"
#     "master/data/snips/"
# )
# for filename in ["train", "valid", "test", "vocab.intent", "vocab.slot"]:
#     path = Path(filename)
#     if not path.exists():
#         print(f"Downloading {filename}...")
#         urlretrieve(SNIPS_DATA_BASE_URL + filename + "?raw=true", path)

In [3]:
lines_train = Path('dataset/train').read_text('utf-8').strip().splitlines()
lines_test = Path('dataset/test').read_text('utf-8').strip().splitlines()
lines_validation = Path('dataset/valid').read_text('utf-8').strip().splitlines()
print(f"First line of train dataset: {lines_train[0]}")

First line of train dataset: Add:O Don:B-entity_name and:I-entity_name Sherri:I-entity_name to:O my:B-playlist_owner Meditate:B-playlist to:I-playlist Sounds:I-playlist of:I-playlist Nature:I-playlist playlist:O <=> AddToPlaylist


In [4]:
def parse_line(line):
    utterance_data, intent_label = line.split(" <=> ")
    items = utterance_data.split()
    words = [item.rsplit(':', 1)[0] for item in items]
    word_labels = [item.rsplit(':', 1)[1] for item in items]
    return {
        'intent_label': intent_label,
        'words': " ".join(words),
        'words_label': " ".join(word_labels),
        'length': len(words)
    }
parse_line(lines_train[0])

{'intent_label': 'AddToPlaylist',
 'words': 'Add Don and Sherri to my Meditate to Sounds of Nature playlist',
 'words_label': 'O B-entity_name I-entity_name I-entity_name O B-playlist_owner B-playlist I-playlist I-playlist I-playlist I-playlist O',
 'length': 12}

In [5]:
data_train = [parse_line(line) for line in lines_train]
data_test = [parse_line(line) for line in lines_test]
data_validation = [parse_line(line) for line in lines_validation]


In [6]:
data_train[:5]

[{'intent_label': 'AddToPlaylist',
  'words': 'Add Don and Sherri to my Meditate to Sounds of Nature playlist',
  'words_label': 'O B-entity_name I-entity_name I-entity_name O B-playlist_owner B-playlist I-playlist I-playlist I-playlist I-playlist O',
  'length': 12},
 {'intent_label': 'AddToPlaylist',
  'words': 'put United Abominations onto my rare groove playlist',
  'words_label': 'O B-entity_name I-entity_name O B-playlist_owner B-playlist I-playlist O',
  'length': 8},
 {'intent_label': 'AddToPlaylist',
  'words': 'add the tune by misato watanabe to the Trapeo playlist',
  'words_label': 'O O B-music_item O B-artist I-artist O O B-playlist O',
  'length': 10},
 {'intent_label': 'AddToPlaylist',
  'words': 'add this artist to my this is miguel bosé playlist',
  'words_label': 'O O B-music_item O B-playlist_owner B-playlist I-playlist I-playlist I-playlist O',
  'length': 10},
 {'intent_label': 'AddToPlaylist',
  'words': 'add heresy and the hotel choir to the evening acoustic play

In [7]:
from transformers import BertTokenizer

model_name = 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)

/media/pk/New Volume empty/Sem6/NLP/ICSF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from tqdm import tqdm
def encode(data)->pd.DataFrame:
    input_ids = []
    attention_masks = []
    token_type_ids = []
    intent_labels = []
    slot_labels = []
    
    for item in tqdm(data):
        encoding = tokenizer(
            item['words'],
            padding='max_length',
            truncation=True,
            max_length=50,
            return_tensors='np'
        )
        input_ids.append(encoding['input_ids'][0])
        attention_masks.append(encoding['attention_mask'][0])
        token_type_ids.append(encoding['token_type_ids'][0])
        intent_labels.append(item['intent_label'])
        slot_labels.append(item['words_label'])
    
    return pd.DataFrame({
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'token_type_ids': token_type_ids,
        'intent_label': intent_labels,
        'slot_label': slot_labels
    })

In [9]:
df_train = encode(data_train)
df_test = encode(data_test)
df_val = encode(data_validation)

100%|██████████| 700/700 [00:00<00:00, 4360.94it/s]


In [10]:
df_train['input_ids'][1]
#print vocab size
#print(f"Vocab size: {tokenizer.vocab_size}")

array([  101,  1508,  1244,   138,  4043,  9204,  1116,  2135,  1139,
        4054, 22482,  1505,  7276,   102,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0])

In [11]:
intent_names = Path('dataset/vocab.intent').read_text('utf-8').split()
intent_map = dict((label, idx) for idx, label in enumerate(intent_names))
intent_map

{'AddToPlaylist': 0,
 'BookRestaurant': 1,
 'GetWeather': 2,
 'PlayMusic': 3,
 'RateBook': 4,
 'SearchCreativeWork': 5,
 'SearchScreeningEvent': 6}

In [12]:
intent_train = df_train['intent_label'].map(intent_map).values
intent_validation = df_val['intent_label'].map(intent_map).values
intent_test = df_test['intent_label'].map(intent_map).values

In [23]:
import torch
from torch import nn
from transformers import BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

from src.models.BiLSTMIntentClassifier import BiLSTMIntentClassifier


Using device: cuda


In [26]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import Adam

def dataloader_from_data(df, intent_labels, batch_size=32, shuffle=True):
    input_ids = torch.tensor(np.stack(df['input_ids'].values))
    attention_masks = torch.tensor(np.stack(df['attention_mask'].values))
    token_type_ids = torch.tensor(np.stack(df['token_type_ids'].values))
    intent_labels = torch.tensor(intent_labels)
    
    dataset = TensorDataset(input_ids, attention_masks, token_type_ids, intent_labels)
    if shuffle:
        sampler = RandomSampler(dataset)
    else:
        sampler = SequentialSampler(dataset)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)
    return dataloader

def train_model(model, train_dataloader, val_dataloader, epochs=5, batch_size=32, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}"):
            input_ids, attention_masks, token_type_ids, labels = [x.to(device) for x in batch]
            optimizer.zero_grad()
            outputs = model(input_ids, attention_masks)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}, Training Loss: {avg_train_loss:.4f}")
        
        model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc=f"Validation Epoch {epoch+1}"):
                input_ids, attention_masks, token_type_ids, labels = [x.to(device) for x in batch]
                outputs = model(input_ids, attention_masks)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        avg_val_loss = total_val_loss / len(val_dataloader)
        val_accuracy = correct / total
        print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

def evaluate_model(model, test_dataloader):
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids, attention_masks, token_type_ids, labels = [x.to(device) for x in batch]
            outputs = model(input_ids, attention_masks)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    test_accuracy = correct / total
    print(f"Test Accuracy: {test_accuracy:.4f}")

In [25]:
print(f"Number of intents: {len(intent_names)}")

model = BiLSTMIntentClassifier(
    hidden_dim=128, 
    output_dim=len(intent_names)
)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

Number of intents: 7
Loading BERT model... This may take a few minutes on first run.
BERT model loaded successfully!
Model created with 109231623 parameters


In [27]:
train_dataloader = dataloader_from_data(df_train, intent_train, batch_size=32, shuffle=True)
test_dataloader = dataloader_from_data(df_test, intent_test, batch_size=32, shuffle=False)
val_dataloader = dataloader_from_data(df_val, intent_validation, batch_size=32, shuffle=False)
train_model(model, train_dataloader, val_dataloader, epochs=1, batch_size=32, lr=1e-3)
evaluate_model(model, test_dataloader)

Training Epoch 1: 100%|██████████| 409/409 [02:05<00:00,  3.25it/s]


Epoch 1, Training Loss: 0.2080


Validation Epoch 1: 100%|██████████| 22/22 [00:06<00:00,  3.36it/s]


Epoch 1, Validation Loss: 0.0945, Validation Accuracy: 0.9671


Testing: 100%|██████████| 22/22 [00:06<00:00,  3.36it/s]

Test Accuracy: 0.9586


In [ ]:
# Option 1: Save only the model weights (recommended)
torch.save(model.state_dict(), 'bilstm_intent_classifier.pth')

# To load it later:
# model = BiLSTMIntentClassifier(hidden_dim=128, output_dim=len(intent_names))
# model.load_state_dict(torch.load('bilstm_intent_classifier.pth'))